In [1]:
import numpy as np

In [2]:
import pandas as pd
labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']

df = pd.read_csv('SEP-28k-Filtered_with_Split_and_Path.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 28177 entries, 0 to 28176
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Unnamed: 0        28177 non-null  int64
 1   Show              28177 non-null  str  
 2   EpId              28177 non-null  int64
 3   ClipId            28177 non-null  int64
 4   SEP28k-T          28177 non-null  str  
 5   SEP28k-D          28177 non-null  str  
 6   Prolongation      28177 non-null  int64
 7   Block             28177 non-null  int64
 8   SoundRep          28177 non-null  int64
 9   WordRep           28177 non-null  int64
 10  Interjection      28177 non-null  int64
 11  NoStutteredWords  28177 non-null  int64
 12  filepath          28177 non-null  str  
dtypes: int64(9), str(4)
memory usage: 2.8 MB


In [3]:

value_counts_table = (
    pd.concat(
        [df[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

value_counts_table

,Prolongation,Block,SoundRep,WordRep,Interjection
0,19631,16207,22563,23556,18519
1,5734,8600,3272,1851,3685
2,2022,2842,1479,1160,2595
3,790,528,863,1610,3378


In [4]:
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [5]:
df_filtered = df[
    df[labels].ge(2).any(axis=1) | df[labels].eq(0).all(axis=1)
].copy() #keep rows with >= 2, and rows with 0 (for fluent)


In [6]:
df_filtered.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath
0,0,HeStutters,0,0,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_0.wav
1,1,HeStutters,0,1,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_1.wav
2,2,HeStutters,0,2,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_2.wav
4,4,HeStutters,0,4,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_4.wav
6,6,HeStutters,0,6,test,test,0,0,0,0,0,3,SEP-28k_CLIP/HeStutters/0/HeStutters_0_6.wav


In [6]:

value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in labels],
        axis=1
    )
)

print(value_counts_table)
two_plus = value_counts_table.loc[[2, 3]].sum()
two_plus

   Prolongation  Block  SoundRep  WordRep  Interjection
0         14557  12790     15778    16207         12340
1          2657   3866      1906     1049          1713
2          2022   2842      1479     1160          2595
3           790    528       863     1610          3378


Prolongation    2812
Block           3370
SoundRep        2342
WordRep         2770
Interjection    5973
dtype: int64

In [7]:
#turn labels to (1,0) ground truths
df_filtered[labels] = (df_filtered[labels] >= 2).astype(np.float32)

In [8]:

df_filtered[labels]

,Prolongation,Block,SoundRep,WordRep,Interjection
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
28172,0.0,0.0,0.0,0.0,1.0
28173,0.0,0.0,1.0,0.0,0.0
28174,0.0,0.0,0.0,0.0,1.0
28175,0.0,0.0,0.0,0.0,1.0


In [9]:
df_filtered["fluent"] = df[labels].eq(0).all(axis=1).astype(int) #add a fluent flag if all disfluencies are 0.

In [10]:
sample_labels = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection', 'fluent']

In [11]:
value_counts_table = (
    pd.concat(
        [df_filtered[col].value_counts().rename(col) for col in sample_labels],
        axis=1
    )
)

print(value_counts_table)


     Prolongation  Block  SoundRep  WordRep  Interjection  fluent
0.0         17214  16656     17684    17256         14053   14022
1.0          2812   3370      2342     2770          5973    6004


In [13]:
#now if we want to upsample, we need to make entries with ONLY that specific label to be upsampled
#similarly for down sample, we need to make entries with ONLY that label to be downsampled. 


In [ ]:
#down sample to 2342      each for better training.

In [12]:
import numpy as np

label_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection', 'fluent']
max_per_label = 6004  

df_shuffled = df_filtered.sample(frac=1, random_state=42).reset_index(drop=True)

label_counts = {label: 0 for label in label_cols}
selected_indices = []

for idx, row in df_shuffled.iterrows():
    row_labels = [label for label in label_cols if row[label] == 1]
    
    # Check if adding this row would exceed any label limit
    if any(label_counts[label] >= max_per_label for label in row_labels):
        continue
    
    selected_indices.append(idx)
    
    for label in row_labels:
        label_counts[label] += 1
    
    # Stop if all labels reached limit
    if all(count >= max_per_label for count in label_counts.values()):
        break

df_downsampled = df_shuffled.loc[selected_indices].reset_index(drop=True)

In [17]:
#At this point, it's going it be a multilabel classfication problem. Each input can have multiple ground truths that are 1 for the 5 disfluencies. 
#If the 5 labels are all 0, that means that it's fluent.

In [21]:
df = df_downsampled.copy() 
df

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath,fluent
0,8750,StrongVoices,19,6,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/19/StrongVoices_19_6...,1
1,9682,StrongVoices,32,28,train,dev,1.0,0.0,0.0,1.0,0.0,0,SEP-28k_CLIP/StrongVoices/32/StrongVoices_32_2...,0
2,8274,StrongVoices,10,70,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/10/StrongVoices_10_7...,1
3,5027,IStutterSoWhat,3,68,dev,train,0.0,0.0,0.0,0.0,1.0,2,SEP-28k_CLIP/IStutterSoWhat/3/IStutterSoWhat_3...,0
4,2564,HeStutters,18,184,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/HeStutters/18/HeStutters_18_184.wav,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20021,15976,StutteringIsCool,24,15,test,test,0.0,1.0,0.0,0.0,0.0,1,SEP-28k_CLIP/StutteringIsCool/24/StutteringIsC...,0
20022,16929,StutteringIsCool,44,11,test,test,1.0,0.0,0.0,0.0,1.0,0,SEP-28k_CLIP/StutteringIsCool/44/StutteringIsC...,0
20023,7540,MyStutteringLife,37,31,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/MyStutteringLife/37/MyStutteringL...,1
20024,1232,HeStutters,8,90,dev,train,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/HeStutters/8/HeStutters_8_90.wav,1


In [ ]:
df['audio_path'] = (
    "../../../archive/Training/" + 
    df['filepath']
)

In [26]:
df.head()

,Unnamed: 0,Show,EpId,ClipId,SEP28k-T,SEP28k-D,Prolongation,Block,SoundRep,WordRep,Interjection,NoStutteredWords,filepath,fluent,audio_path
0,8750,StrongVoices,19,6,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/19/StrongVoices_19_6...,1,../../../archive/Training/SEP-28k_CLIP/StrongV...
1,9682,StrongVoices,32,28,train,dev,1.0,0.0,0.0,1.0,0.0,0,SEP-28k_CLIP/StrongVoices/32/StrongVoices_32_2...,0,../../../archive/Training/SEP-28k_CLIP/StrongV...
2,8274,StrongVoices,10,70,train,dev,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/StrongVoices/10/StrongVoices_10_7...,1,../../../archive/Training/SEP-28k_CLIP/StrongV...
3,5027,IStutterSoWhat,3,68,dev,train,0.0,0.0,0.0,0.0,1.0,2,SEP-28k_CLIP/IStutterSoWhat/3/IStutterSoWhat_3...,0,../../../archive/Training/SEP-28k_CLIP/IStutte...
4,2564,HeStutters,18,184,test,test,0.0,0.0,0.0,0.0,0.0,3,SEP-28k_CLIP/HeStutters/18/HeStutters_18_184.wav,1,../../../archive/Training/SEP-28k_CLIP/HeStutt...


In [27]:
import os
test_path = df['audio_path'].iloc[0]
print(f"Checking at: {os.path.abspath(test_path)}")
if not os.path.exists(test_path):
    print("File not found.")
else:
    print("File found.")

Checking at: c:\MyStuff\SJSU\Spring 2026\Ling 230\DisfluencyProject\disfluency_detection\archive\Training\SEP-28k_CLIP\StrongVoices\19\StrongVoices_19_6.wav
File found.


In [28]:
import pandas as pd
import torch
import librosa
import os
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm # visualization
import numpy as np 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # for training
from sklearn.utils.class_weight import compute_class_weight # class weights
from sklearn.model_selection import train_test_split # split data
from sklearn.preprocessing import LabelEncoder #transform labels
import matplotlib.pyplot as plt # visualization
import seaborn as sns # visualization
from sklearn.metrics import confusion_matrix, classification_report # visualization
import copy # copy and save model 
import math
import gc

In [29]:
# Set up the Whisper Model
model_name = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
model = WhisperModel.from_pretrained(model_name)

# find Mac GPU, also allowed for NVIDIA GPU
if torch.backends.mps.is_available():
    device = "mps"
    print("Using Mac GPU (MPS) for acceleration!")
elif torch.cuda.is_available():
    device = "cuda"
    print("Using NVIDIA GPU (CUDA) for acceleration!")
else:
    device = "cpu"
    print("Using CPU. (No GPU detected)")

model.to(device)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using NVIDIA GPU (CUDA) for acceleration!


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 512, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(512, 512, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 512)
    (layers): ModuleList(
      (0-5): 6 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=512, out_features=512, bias=False)
          (v_proj): Linear(in_features=512, out_features=512, bias=True)
          (q_proj): Linear(in_features=512, out_features=512, bias=True)
          (out_proj): Linear(in_features=512, out_features=512, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=512, out_features=2048, bias=True)
        (fc2): Linear(in_features=2048, out_features=512, bias=True)
        (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )


In [30]:
def get_whisper_embedding_sequence(audio_path):
    if not os.path.exists(audio_path): return None
    try:
        # Load audio at 16k Hz
        speech_array, sampling_rate = librosa.load(audio_path, sr=16000)
        inputs = feature_extractor(speech_array, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)
        
        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            
        # remove squeeze dimension
        embedding = encoder_outputs.last_hidden_state.squeeze(0).cpu().numpy()
        embedding = embedding[:150]

        return embedding 
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
import numpy as np
from tqdm import tqdm

TARGET_T = 150
TARGET_D = 512  # change to 768 if whisper-small
N = len(df)

print("Creating memory-mapped .npy file...")

# Create final .npy file directly
embeddings = np.lib.format.open_memmap(
    "whisper_embeddings2k.npy",
    mode="w+",
    dtype=np.float32,
    shape=(N, TARGET_T, TARGET_D),
)

valid_mask = np.zeros(N, dtype=bool)

print("Processing audio files...")

for idx, path in tqdm(enumerate(df["audio_path"]), total=N):
    emb = get_whisper_embedding_sequence(path)

    if emb is None or emb.shape != (TARGET_T, TARGET_D):
        continue

    embeddings[idx] = emb
    valid_mask[idx] = True

# Flush everything to disk
del embeddings  # important — ensures file is written

print("Initial save complete.")

In [ ]:
import numpy as np
import os

valid_indices = np.where(valid_mask)[0]

in_path = "whisper_embeddings2k.npy"
out_path = "whisper_embeddings2k.npy"

# Load memory-mapped file (read-only)
full = np.load(in_path, mmap_mode="r")

# Create compact final file (NEW NAME)
compact = np.lib.format.open_memmap(
    out_path,
    mode="w+",
    dtype=np.float32,
    shape=(len(valid_indices), TARGET_T, TARGET_D),
)

for i, idx in enumerate(valid_indices):
    compact[i] = full[idx]

# IMPORTANT: release file handles on Windows
del compact
del full

clean_df = df.iloc[valid_indices].copy()
clean_df.to_csv("SEP-28k_with_paths2k.csv", index=False)

print("Final compact file saved:", out_path)
print("Final shape:", (len(valid_indices), TARGET_T, TARGET_D))

# Optional: replace original AFTER both are closed
os.replace(out_path, in_path)

Final compact file saved: whisper_embeddings_no_sample_compact.npy
Final shape: (20026, 150, 512)
